In [1]:
# ============================================================
# M1 — LOAD FINAL SCALED 2025 NETWORK (NO-HUB, FULL YEAR)
# ============================================================

import pypsa
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS (HARD, EXPLICIT — NO KERNEL STATE)
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

base_fn = (
    root
    / "results"
    / "networks"
    / "base_s_50_elec_scaled_2025.nc"
)

print("M1 loading network from:")
print(base_fn)

# ------------------------------------------------------------
# LOAD NETWORK (FULL YEAR, 8760)
# ------------------------------------------------------------
n = pypsa.Network(base_fn)
print("Loaded:", base_fn.name)
print("Snapshots:", len(n.snapshots))

# ------------------------------------------------------------
# SAFETY — ENSURE CARRIER TABLE IS NETCDF / PYPSA SAFE
# ------------------------------------------------------------
if "color" in n.carriers.columns:
    n.carriers["color"] = n.carriers["color"].astype(str)

# Ensure all generator carriers exist in n.carriers
n.generators["carrier"] = n.generators["carrier"].astype(str)
missing_carriers = set(n.generators.carrier.unique()) - set(n.carriers.index)

if missing_carriers:
    print("INFO: Adding missing carriers to n.carriers:", sorted(missing_carriers))
    default_row = {c: 0 for c in n.carriers.columns}
    for mc in sorted(missing_carriers):
        n.carriers.loc[mc] = default_row

# ------------------------------------------------------------
# NO-HUB / DISPATCH-ONLY (ABSOLUTE)
# ------------------------------------------------------------
for df_name in ["generators", "links", "lines", "storage_units"]:
    if hasattr(n, df_name):
        df = getattr(n, df_name)
        if "p_nom_extendable" in df.columns:
            df["p_nom_extendable"] = False

if hasattr(n, "lines") and "s_nom_extendable" in n.lines.columns:
    n.lines["s_nom_extendable"] = False

print("✓ Expansion disabled (No-Hub mode)")

# ------------------------------------------------------------
# STORAGE — NON-CYCLIC, DAILY ROLLING COMPATIBLE
# ------------------------------------------------------------
if hasattr(n, "storage_units") and not n.storage_units.empty:
    if "cyclic_state_of_charge" in n.storage_units.columns:
        n.storage_units["cyclic_state_of_charge"] = False
    if "e_initial" not in n.stores.columns:
        n.stores["e_initial"] = 0.0

if hasattr(n, "stores") and not n.stores.empty:
    if "cyclic_state_of_charge" in n.stores.columns:
        n.stores["cyclic_state_of_charge"] = False
    if "e_initial" not in n.stores.columns:
        n.stores["e_initial"] = 0.0

print("✓ Storage set to non-cyclic with explicit initial states")

# ------------------------------------------------------------
# LOAD SHEDDING — LAST RESORT (CONSISTENT WITH SCALED_2025)
# ------------------------------------------------------------
LS_COST = 10_000.0  # €/MWh

if "load_shedding" not in n.carriers.index:
    n.carriers.loc["load_shedding"] = {c: 0 for c in n.carriers.columns}

missing_ls = [b for b in n.buses.index if f"load_shed_{b}" not in n.generators.index]

for b in missing_ls:
    n.add(
        "Generator",
        name=f"load_shed_{b}",
        bus=b,
        carrier="load_shedding",
        p_nom=1e6,
        p_min_pu=0.0,
        p_max_pu=1.0,
        marginal_cost=LS_COST,
    )

print("✓ Load shedding generators ensured:", len(missing_ls))

# ------------------------------------------------------------
# SNAPSHOT BACKUP (FOR M2 / M3 DAILY SLICING)
# ------------------------------------------------------------
FULL_SNAPSHOTS = n.snapshots.copy()

# ------------------------------------------------------------
# OUTPUT DIRECTORIES (USED BY M3–M5)
# ------------------------------------------------------------
out_root = root / "results" / "daily_runs"
by_day   = out_root / "by_day"
by_month = out_root / "by_month"

by_day.mkdir(parents=True, exist_ok=True)
by_month.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# SOLVER
# ------------------------------------------------------------
SOLVER = "cbc"   # robust for daily rolling
print("Solver set to:", SOLVER)

# ------------------------------------------------------------
# SANITY CHECK — SEASONAL HYDRO WATER VALUE EXISTS
# ------------------------------------------------------------
assert hasattr(n, "storage_units_t"), "storage_units_t missing"
assert "marginal_cost" in n.storage_units_t, "Seasonal hydro marginal cost missing"

print("✓ Seasonal hydro water value detected")
print("M1 READY — proceed to M2")


M1 loading network from:
C:\Users\franc\pypsa\pypsa-eur\results\networks\base_s_50_elec_scaled_2025.nc


INFO:pypsa.io:Imported network base_s_50_elec_scaled_2025.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


Loaded: base_s_50_elec_scaled_2025.nc
Snapshots: 8760
✓ Expansion disabled (No-Hub mode)
✓ Storage set to non-cyclic with explicit initial states
✓ Load shedding generators ensured: 0
Solver set to: cbc
✓ Seasonal hydro water value detected
M1 READY — proceed to M2


In [2]:
# ============================================================
# M2 — Helpers for one-day solve, metrics, and rolling states
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Country mappings
# ------------------------------------------------------------

bus_country  = n.buses.country
gen_country  = n.generators.bus.map(bus_country)
load_country = n.loads.bus.map(bus_country)

MAIN_COUNTRIES = ["DK", "NL", "BE", "DE", "FR", "NO", "PL", "SE", "PT"]

# ------------------------------------------------------------
# Defensive: ensure all generator carriers exist in n.carriers
# (NetCDF + PyPSA consistency safety)
# ------------------------------------------------------------

n.generators["carrier"] = n.generators["carrier"].astype(str)
missing_carriers = set(n.generators.carrier.unique()) - set(n.carriers.index)

if missing_carriers:
    print("INFO (M2): adding missing carriers:", sorted(missing_carriers))
    default_row = {c: 0 for c in n.carriers.columns}
    for mc in missing_carriers:
        n.carriers.loc[mc] = default_row

# ------------------------------------------------------------
# Snapshot helpers
# ------------------------------------------------------------

def day_index(ts: pd.Timestamp) -> pd.DatetimeIndex:
    """Return hourly snapshots for a given day that exist in FULL_SNAPSHOTS."""
    d = pd.Timestamp(ts).normalize()
    hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq="h")
    return hours.intersection(FULL_SNAPSHOTS)

# ------------------------------------------------------------
# Rolling storage states (ONLY for stores / pumped hydro)
# Reservoir hydro is handled via p_max_pu (Option A)
# ------------------------------------------------------------

def roll_storage_initials_from_solution():
    """Carry last-hour storage states into next day."""

    # StorageUnits (pumped hydro, batteries)
    if hasattr(n, "storage_units_t") and "state_of_charge" in n.storage_units_t:
        last_soc = (
            n.storage_units_t.state_of_charge.iloc[-1]
            .reindex(n.storage_units.index)
            .fillna(0.0)
        )
        if "state_of_charge_initial" not in n.storage_units.columns:
            n.storage_units["state_of_charge_initial"] = 0.0
        n.storage_units["state_of_charge_initial"] = last_soc

    # Stores (if any)
    if hasattr(n, "stores_t") and "e" in n.stores_t:
        last_e = (
            n.stores_t.e.iloc[-1]
            .reindex(n.stores.index)
            .fillna(0.0)
        )
        if "e_initial" not in n.stores.columns:
            n.stores["e_initial"] = 0.0
        n.stores["e_initial"] = last_e

# ------------------------------------------------------------
# Metric extraction (Option A compatible)
# ------------------------------------------------------------

def extract_day_metrics(day_idx: pd.DatetimeIndex) -> dict:
    """Collect daily KPIs (mean power, prices, curtailment, CO2)."""

    out = {}

    loads_ts = n.loads_t.p_set.loc[day_idx]      # MW
    gens_ts  = n.generators_t.p.loc[day_idx]     # MW

    # -----------------
    # Demand (mean MW)
    # -----------------

    out["demand_EU_MW"] = float(loads_ts.sum(axis=1).mean())

    for c in MAIN_COUNTRIES:
        mask = (load_country == c)
        if mask.any():
            out[f"demand_{c}_MW"] = float(
                loads_ts.loc[:, mask].sum(axis=1).mean()
            )

    # -----------------
    # Generation (mean MW)
    # -----------------

    out["generation_EU_MW"] = float(gens_ts.sum(axis=1).mean())

    for c in MAIN_COUNTRIES:
        mask = (gen_country == c)
        if mask.any():
            out[f"generation_{c}_MW"] = float(
                gens_ts.loc[:, mask].sum(axis=1).mean()
            )

    # -----------------
    # Curtailment (VRE only, mean MW)
    # -----------------

    if hasattr(n.generators_t, "p_max_pu"):
        avail_ts = n.generators_t.p_max_pu.loc[day_idx].mul(
            n.generators.p_nom, axis=1
        )
        curtail_ts = (avail_ts - gens_ts).clip(lower=0.0)

        for c in MAIN_COUNTRIES:
            mask = (gen_country == c)
            if mask.any():
                out[f"curtailment_{c}_MW"] = float(
                    curtail_ts.loc[:, mask].sum(axis=1).mean()
                )

    # -----------------
    # Prices (€/MWh)
    # -----------------

    if hasattr(n, "buses_t") and "marginal_price" in n.buses_t:
        prices_ts = n.buses_t.marginal_price.loc[day_idx]

        out["price_EU_EUR_per_MWh"] = float(prices_ts.stack().mean())

        for c in MAIN_COUNTRIES:
            mask = (bus_country == c)
            if mask.any():
                out[f"price_{c}_EUR_per_MWh"] = float(
                    prices_ts.loc[:, mask].mean(axis=1).mean()
                )

    # -----------------
    # Load shedding (mean MW)
    # -----------------

    if (n.generators.carrier == "load_shedding").any():
        shed_ts = gens_ts.loc[:, n.generators.carrier == "load_shedding"]
        out["load_shed_MW"] = float(shed_ts.sum(axis=1).mean())
    else:
        out["load_shed_MW"] = 0.0

    # -----------------
    # CO₂ emissions (t/day)
    # -----------------

    if "co2_emissions" in n.carriers.columns:
        eff = n.generators.efficiency.replace(0, np.nan).fillna(1.0)
        co2_rate = (
            n.generators.carrier
            .map(n.carriers.co2_emissions)
            .fillna(0.0)
        )

        emis_ts = gens_ts.divide(eff, axis=1).multiply(co2_rate, axis=1)
        out["tCO2_EU"] = float(emis_ts.sum().sum())

        by_country = emis_ts.sum().groupby(gen_country).sum()
        for c, v in by_country.items():
            out[f"tCO2_{c}"] = float(v)

    return out

# ------------------------------------------------------------
# One-day solve wrapper
# ------------------------------------------------------------

def solve_one_day(
    d: pd.Timestamp,
    solver: str = None,
    threads: int = 4,
    logdir=None,
) -> dict:
    """Solve OPF for one day and return metrics."""

    idx = day_index(d)
    if len(idx) == 0:
        return {"note": "no snapshots"}

    n.set_snapshots(idx)

    kwargs = {}
    if logdir is not None:
        logdir.mkdir(parents=True, exist_ok=True)
        tag = idx[0].date()
        kwargs["log_fn"]      = str(logdir / f"{solver or SOLVER}_{tag}.log")
        kwargs["problem_fn"]  = str(logdir / f"{solver or SOLVER}_{tag}.lp")
        kwargs["solution_fn"] = str(logdir / f"{solver or SOLVER}_{tag}.sol")

    status, cond = n.optimize(
        solver_name=(solver or SOLVER),
        threads=threads,
        progress=True,
        **kwargs,
    )

    if status != "ok":
        return {"note": f"solve status {status} ({cond})"}

    metrics = extract_day_metrics(idx)
    roll_storage_initials_from_solution()
    return metrics

# ------------------------------------------------------------
# Smoke test
# ------------------------------------------------------------

d0 = pd.Timestamp("2013-01-01")
print("Testing day:", d0.date())
print(solve_one_day(d0, solver=SOLVER, threads=4))
print("done")


Testing day: 2013-01-01


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|███████████████████████████████████████████████████| 10/10 [00:00<00:00, 77.86it/s]
INFO:linopy.io: Writing time: 0.67s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-7tfhts0k.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-9b0xmfnh.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7204 (-62064) rows, 16481 (-16979) columns and 33164 (-84254) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.11096999%) - largest zero change 0.00086368374
0  Obj 62849.893 Primal inf 16252772 (2033) Dual i

{'demand_EU_MW': 321042.7592837016, 'demand_DK_MW': 3518.5416673024497, 'demand_NL_MW': 11271.708331425985, 'demand_BE_MW': 7919.916669845581, 'demand_DE_MW': 41525.18316332499, 'demand_FR_MW': 54434.8333346049, 'demand_NO_MW': 16243.166668256124, 'demand_PL_MW': 13341.333334287008, 'demand_SE_MW': 15982.95834604899, 'demand_PT_MW': 4550.0, 'generation_EU_MW': 311288.1975072241, 'generation_DK_MW': 5905.060926220834, 'generation_NL_MW': 8913.751882458333, 'generation_BE_MW': 8946.79924, 'generation_DE_MW': 67927.355015, 'generation_FR_MW': 49705.97512460625, 'generation_NO_MW': 1400.184262075, 'generation_PL_MW': 18494.590150333333, 'generation_SE_MW': 13739.469548291665, 'generation_PT_MW': 3609.3831658875, 'curtailment_DK_MW': 2.215136321459532e-05, 'curtailment_NL_MW': 2.8638650851083487e-05, 'curtailment_BE_MW': 2.3205482718206365e-05, 'curtailment_DE_MW': 0.00021633808294711324, 'curtailment_FR_MW': 7.183809820919193e-05, 'curtailment_NO_MW': 9.474163786641713e-06, 'curtailment_PL

In [10]:
# ============================================================
# FINAL BASELINE M3 — FULL EXPORT (DAILY + HOURLY)
# COUNTRY-CORRECT: DEMAND / GENERATION / PRICE / LOAD SHED
# ============================================================

import pandas as pd
import numpy as np
import pypsa
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
months = [f"2013-{m:02d}" for m in range(1, 13)]

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
base_fn  = root / "results" / "networks" / "base_s_50_elec_scaled_2025.nc"
out_root = root / "results" / "daily_runs"
out_root.mkdir(parents=True, exist_ok=True)

SOLVER = "cbc"

# ------------------------------------------------------------
# SNAPSHOT HELPER
# ------------------------------------------------------------
def day_index(ts, full_snapshots):
    d = pd.Timestamp(ts).normalize()
    hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq="h")
    return hours.intersection(full_snapshots)

print("=== FINAL BASELINE M3 — COUNTRY-CORRECT ===")

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
for month in months:
    print(f"\n=== Month {month} ===")

    month_start = pd.Timestamp(f"{month}-01")
    month_end   = month_start + pd.offsets.MonthEnd(1)
    days = pd.date_range(month_start, month_end, freq="D")

    rows_daily  = []
    rows_hourly = []

    last_soc = None
    last_e   = None

    for d in days:
        n = pypsa.Network(base_fn)
        FULL_SNAPSHOTS = n.snapshots.copy()

        idx = day_index(d, FULL_SNAPSHOTS)
        if len(idx) == 0:
            continue

        print(f"→ {d.date()} ({len(idx)} h)")

        # ----------------------------------------------------
        # DISPATCH ONLY
        # ----------------------------------------------------
        for comp in ["generators", "links", "lines", "storage_units"]:
            if hasattr(n, comp):
                df = getattr(n, comp)
                if "p_nom_extendable" in df.columns:
                    df["p_nom_extendable"] = False

        if "s_nom_extendable" in n.lines.columns:
            n.lines["s_nom_extendable"] = False

        # ----------------------------------------------------
        # STORAGE (ROLLING)
        # ----------------------------------------------------
        if not n.storage_units.empty:
            n.storage_units["cyclic_state_of_charge"] = False
            if "state_of_charge_initial" not in n.storage_units.columns:
                n.storage_units["state_of_charge_initial"] = 0.0

        if not n.stores.empty:
            n.stores["cyclic_state_of_charge"] = False
            if "e_initial" not in n.stores.columns:
                n.stores["e_initial"] = 0.0

        if last_soc is not None and not n.storage_units.empty:
            n.storage_units["state_of_charge_initial"] = (
                last_soc.reindex(n.storage_units.index).fillna(0.0)
            )

        if last_e is not None and not n.stores.empty:
            n.stores["e_initial"] = (
                last_e.reindex(n.stores.index).fillna(0.0)
            )

        # ----------------------------------------------------
        # LOAD SHEDDING
        # ----------------------------------------------------
        if "load_shedding" not in n.carriers.index:
            n.carriers.loc["load_shedding"] = {c: 0 for c in n.carriers.columns}

        for b in n.buses.index:
            g = f"load_shed_{b}"
            if g not in n.generators.index:
                n.add(
                    "Generator",
                    name=g,
                    bus=b,
                    carrier="load_shedding",
                    p_nom=1e6,
                    marginal_cost=10_000.0,
                )

        # ----------------------------------------------------
        # SOLVE
        # ----------------------------------------------------
        n.set_snapshots(idx)
        status, cond = n.optimize(
            solver_name=SOLVER,
            threads=4,
            progress=False,
        )

        if status != "ok":
            rows_daily.append({"date": d.date(), "note": f"{status} ({cond})"})
            continue

        # ----------------------------------------------------
        # SAVE STORAGE STATE
        # ----------------------------------------------------
        if not n.storage_units.empty:
            last_soc = n.storage_units_t.state_of_charge.iloc[-1].copy()

        if not n.stores.empty:
            last_e = n.stores_t.e.iloc[-1].copy()

        # ----------------------------------------------------
        # BASE TIME SERIES
        # ----------------------------------------------------
        load_p = n.loads_t.p_set.loc[idx]
        gen_p  = n.generators_t.p.loc[idx]
        prices = n.buses_t.marginal_price.loc[idx]

        total_demand_MW     = load_p.sum(axis=1)
        total_generation_MW = gen_p.sum(axis=1)

        hourly_df = pd.DataFrame({
            "snapshot": idx,
            "date": idx.normalize(),
            "hour": idx.hour,
            "total_demand_MW": total_demand_MW.values,
            "total_generation_MW": total_generation_MW.values,
        })

        # ----------------------------------------------------
        # COUNTRY MAPS
        # ----------------------------------------------------
        bus_country = n.buses.country
        load_country = n.loads.bus.map(bus_country)
        gen_country  = n.generators.bus.map(bus_country)

        # ----------------------------------------------------
        # DEMAND BY COUNTRY
        # ----------------------------------------------------
        for c in MAIN_COUNTRIES:
            mask = (load_country == c)
            if mask.any():
                hourly_df[f"demand_{c}_MW"] = load_p.loc[:, mask].sum(axis=1).values
            else:
                hourly_df[f"demand_{c}_MW"] = 0.0

        # ----------------------------------------------------
        # GENERATION BY COUNTRY
        # ----------------------------------------------------
        for c in MAIN_COUNTRIES:
            mask = (gen_country == c)
            if mask.any():
                hourly_df[f"generation_{c}_MW"] = gen_p.loc[:, mask].sum(axis=1).values
            else:
                hourly_df[f"generation_{c}_MW"] = 0.0

        # -------------------------------
        # PRICE BY COUNTRY (LOAD-WEIGHTED, LOAD BUSES ONLY)
        # -------------------------------
        prices_bus = n.buses_t.marginal_price.loc[idx]

        # load per bus (MW)
        load_by_bus = load_p.T.groupby(n.loads.bus).sum().T

        for c in MAIN_COUNTRIES:
            buses_c = bus_country[bus_country == c].index
            buses_c = buses_c.intersection(load_by_bus.columns)

            if len(buses_c) == 0:
                hourly_df[f"price_{c}_EUR_per_MWh"] = np.nan
                continue

            load_c = load_by_bus[buses_c]
            price_c = prices_bus[buses_c]

            denom = load_c.sum(axis=1).replace(0, np.nan)
            p_weighted = (price_c * load_c).sum(axis=1) / denom

            hourly_df[f"price_{c}_EUR_per_MWh"] = p_weighted.values


        # ----------------------------------------------------
        # LOAD SHEDDING (EU)
        # ----------------------------------------------------
        if (n.generators.carrier == "load_shedding").any():
            shed = gen_p.loc[:, n.generators.carrier == "load_shedding"].sum(axis=1)
            hourly_df["load_shed_MW"] = shed.values
        else:
            hourly_df["load_shed_MW"] = 0.0

        # ----------------------------------------------------
        # GENERATION BY CARRIER (EU)
        # ----------------------------------------------------
        gen_by_carrier = gen_p.T.groupby(n.generators.carrier).sum().T
        for car in gen_by_carrier.columns:
            hourly_df[f"gen_{car}_MW"] = gen_by_carrier[car].values

        rows_hourly.append(hourly_df)

        # ----------------------------------------------------
        # DAILY METRICS
        # ----------------------------------------------------
        rows_daily.append({
            "date": d.date(),
            "demand_MWh": total_demand_MW.sum(),
            "generation_MWh": total_generation_MW.sum(),
            "avg_price_all_EUR_per_MWh": prices.stack().mean(),
            "load_shed_MWh": hourly_df["load_shed_MW"].sum(),
        })

    # --------------------------------------------------------
    # SAVE MONTH
    # --------------------------------------------------------
    out_dir = out_root / month
    out_dir.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(rows_daily).set_index("date").to_csv(
        out_dir / f"daily_metrics_{month}.csv"
    )

    if rows_hourly:
        pd.concat(rows_hourly, ignore_index=True).to_csv(
            out_dir / f"hourly_power_{month}.csv",
            index=False
        )

    print(f"✓ Saved {month}")

print("\n=== M3 BASELINE COMPLETED SUCCESSFULLY ===")


=== FINAL BASELINE M3 — COUNTRY-CORRECT ===

=== Month 2013-01 ===


INFO:pypsa.io:Imported network base_s_50_elec_scaled_2025.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


→ 2013-01-01 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.62s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-ylydfapd.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-saqfz6_o.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7204 (-62064) rows, 16481 (-16979) columns and 33164 (-84254) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.11096999%) - largest zero change 0.00086368374
0  Obj 62849.893 Primal inf 16252772 (2033) Dual inf 952.87582 (24)
219  Obj -58.977363 Primal inf 20622675 (1984)
438  Obj 11904732 Primal inf 27144702 (2019)
657  Obj 16870691 Primal inf 68425777 (2044)

→ 2013-01-02 (24 h)


...

→ 2013-12-30 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.59s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-tx5f0g9f.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-c5u5jno5.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7196 (-62072) rows, 16455 (-17005) columns and 33128 (-84290) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.099417293%) - largest zero change 0.00086368374
0  Obj 7775867.2 Primal inf 16674366 (2048) Dual inf 20401.761 (26)
218  Obj 256274.59 Primal inf 36607144 (2014)
436  Obj 11400371 Primal inf 25649688 (2039)
654  Obj 22899066 Primal inf 31234624 (2076)

→ 2013-12-31 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.68s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-92tx5ajz.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-8l7jl5sv.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7207 (-62061) rows, 16475 (-16985) columns and 33165 (-84253) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.10083713%) - largest zero change 0.00086368374
0  Obj 175036.23 Primal inf 16589126 (2060) Dual inf 952.87582 (24)
219  Obj 577.96282 Primal inf 21701343 (2008)
438  Obj 10314257 Primal inf 28002375 (2034)
657  Obj 18276029 Primal inf 31608595 (2073)


✓ Saved 2013-12

=== M3 BASELINE COMPLETED SUCCESSFULLY ===


In [11]:
# ============================================================
# FINAL BASELINE M4 — YEAR AGGREGATION (2013)
# NO-HUB SCENARIO
# FULL VERSION — DEFINITIVE FIX
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
out_root = root / "results" / "daily_runs"
baseline_dir = out_root / "baseline_nohub_2013"
baseline_dir.mkdir(parents=True, exist_ok=True)

written_files = []

print("M4 writing to:", baseline_dir.resolve())

# ------------------------------------------------------------
# LOAD DAILY + HOURLY FILES
# ------------------------------------------------------------

months = [f"2013-{m:02d}" for m in range(1, 13)]

daily_frames = []
hourly_frames = []

for mm in months:
    dfn = out_root / mm / f"daily_metrics_{mm}.csv"
    hfn = out_root / mm / f"hourly_power_{mm}.csv"

    if dfn.exists():
        daily_frames.append(
            pd.read_csv(dfn, parse_dates=["date"]).set_index("date")
        )

    if hfn.exists():
        hourly_frames.append(
            pd.read_csv(hfn, parse_dates=["snapshot", "date"])
        )

if not daily_frames or not hourly_frames:
    raise RuntimeError("Run baseline M3 first.")

daily = pd.concat(daily_frames).sort_index()
hourly = pd.concat(hourly_frames, ignore_index=True)

print(f"Loaded {len(daily)} daily rows.")
print(f"Loaded {len(hourly)} hourly rows.")

# ------------------------------------------------------------
# EU KPIs (FROM DAILY — CORRECT)
# ------------------------------------------------------------

demand_total = daily["demand_MWh"].sum()
generation_total = daily["generation_MWh"].sum()

variable_cost_total = (
    daily["variable_cost_EUR"].sum()
    if "variable_cost_EUR" in daily.columns else 0.0
)

avg_price_mean = (
    daily["avg_price_all_EUR_per_MWh"].mean()
    if "avg_price_all_EUR_per_MWh" in daily.columns else np.nan
)

total_load_shed = (
    daily["load_shed_MWh"].sum()
    if "load_shed_MWh" in daily.columns else 0.0
)

annual_summary_df = pd.DataFrame([{
    "demand_MWh_total": demand_total,
    "generation_MWh_total": generation_total,
    "variable_cost_EUR_total": variable_cost_total,
    "avg_price_EUR_per_MWh_mean": avg_price_mean,
    "total_load_shed_MWh": total_load_shed,
}])

# ------------------------------------------------------------
# GENERATION MIX (EU — FROM HOURLY)
# ------------------------------------------------------------

gen_cols = [c for c in hourly.columns if c.startswith("gen_") and c.endswith("_MW")]

gen_by_carrier = (
    hourly[gen_cols]
    .sum()
    .rename(lambda c: c.replace("gen_", "").replace("_MW", ""))
)

gen_by_carrier = gen_by_carrier.drop(
    [c for c in gen_by_carrier.index if "load_shedding" in c],
    errors="ignore"
)

gen_mix_table = pd.DataFrame({
    "MWh": gen_by_carrier,
    "%_of_served_generation": 100.0 * gen_by_carrier / gen_by_carrier.sum()
})

# ------------------------------------------------------------
# SAVE STANDARD M4 FILES
# ------------------------------------------------------------

f = baseline_dir / "baseline_nohub_2013_daily.csv"
daily.to_csv(f); written_files.append(f.name)

f = baseline_dir / "baseline_nohub_2013_summary.csv"
annual_summary_df.to_csv(f, index=False); written_files.append(f.name)

f = baseline_dir / "baseline_nohub_2013_generation_mix.csv"
gen_mix_table.to_csv(f); written_files.append(f.name)

f = baseline_dir / "baseline_nohub_2013_curtailment.csv"
pd.Series(dtype=float).to_csv(f); written_files.append(f.name)

f = baseline_dir / "baseline_nohub_2013_net_exports.csv"
pd.Series(dtype=float).to_csv(f); written_files.append(f.name)

# ============================================================
# COUNTRIES MAIN — DEFINITIVE (WITH LOAD SHEDDING FIX)
# ============================================================

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]
rows = []

# ----------------------------------------
# Identify load shedding hours (EU)
# ----------------------------------------
if "gen_load_shedding_MW" in hourly.columns:
    shed_ts = hourly["gen_load_shedding_MW"]
    shedding_hours = shed_ts[shed_ts > 0].index
else:
    shed_ts = pd.Series(0.0, index=hourly.index)
    shedding_hours = []

# ----------------------------------------
# Precompute demand during shedding hours
# ----------------------------------------
demand_during_shed = {}
total_demand_shed = 0.0

for c in MAIN_COUNTRIES:
    dcol = f"demand_{c}_MW"
    if dcol in hourly.columns and len(shedding_hours) > 0:
        val = hourly.loc[shedding_hours, dcol].sum()
    else:
        val = 0.0
    demand_during_shed[c] = val
    total_demand_shed += val

# ----------------------------------------
# Build table
# ----------------------------------------
for c in MAIN_COUNTRIES + ["EU"]:

    if c == "EU":
        demand = demand_total
        generation = generation_total
        price = avg_price_mean
        load_shed = total_load_shed

    else:
        dcol = f"demand_{c}_MW"
        gcol = f"generation_{c}_MW"
        pcol = f"price_{c}_EUR_per_MWh"

        demand = hourly[dcol].sum() if dcol in hourly.columns else 0.0
        generation = hourly[gcol].sum() if gcol in hourly.columns else 0.0

        # Load-weighted average price
        if dcol in hourly.columns and pcol in hourly.columns:
            w = hourly[dcol].replace(0, np.nan)

            mask = w > 1e-3
            price = (hourly.loc[mask, pcol] * w.loc[mask]).sum() / w.loc[mask].sum()
        else:
            price = np.nan

        # --------------------------------
        # Load shedding allocation
        # --------------------------------
        if total_demand_shed > 0:
            load_shed = (
                total_load_shed
                * demand_during_shed[c]
                / total_demand_shed
            )
        else:
            load_shed = 0.0

    rows.append({
        "country": c,
        "annual_demand_MWh": demand,
        "annual_generation_MWh": generation,
        "avg_price_EUR_per_MWh": price,
        "load_shed_MWh": load_shed,
    })

countries_main = pd.DataFrame(rows).set_index("country")

f = baseline_dir / "baseline_nohub_2013_countries_main.csv"
countries_main.to_csv(f); written_files.append(f.name)


# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print("\n✔ BASELINE M4 COMPLETE — FILES WRITTEN / UPDATED:")
for fn in written_files:
    print(" -", fn)

print("\n=== BASELINE M4 COMPLETED SUCCESSFULLY ===")


M4 writing to: C:\Users\franc\pypsa\pypsa-eur\results\daily_runs\baseline_nohub_2013
Loaded 365 daily rows.
Loaded 8760 hourly rows.

✔ BASELINE M4 COMPLETE — FILES WRITTEN / UPDATED:
 - baseline_nohub_2013_daily.csv
 - baseline_nohub_2013_summary.csv
 - baseline_nohub_2013_generation_mix.csv
 - baseline_nohub_2013_curtailment.csv
 - baseline_nohub_2013_net_exports.csv
 - baseline_nohub_2013_countries_main.csv

=== BASELINE M4 COMPLETED SUCCESSFULLY ===


In [19]:
# ============================================================
# FINAL BASELINE M5 — CALENDAR 8 CATEGORIES (2013)
# DEFINITIVE VERSION — COUNTRY PRICE FIX
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
out_root = root / "results" / "daily_runs"

clusters_dir = out_root / "clusters" / "calendar_8cats_2013"
clusters_dir.mkdir(parents=True, exist_ok=True)

baseline_dir = out_root / "baseline_nohub_2013"
baseline_dir.mkdir(parents=True, exist_ok=True)

written_files = []

# ------------------------------------------------------------
# LOAD HOURLY FILES
# ------------------------------------------------------------
frames = []

for m in range(1, 13):
    mm = f"2013-{m:02d}"
    fn = out_root / mm / f"hourly_power_{mm}.csv"
    if not fn.exists():
        raise FileNotFoundError(fn)

    df = pd.read_csv(fn, parse_dates=["snapshot", "date"])
    df["hour"] = df["snapshot"].dt.hour
    frames.append(df)

hourly = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(hourly)} hourly rows")

# ------------------------------------------------------------
# SEASON FUNCTION (2013)
# ------------------------------------------------------------
def season_2013(d):
    if date(2013,1,1) <= d <= date(2013,3,19):
        return "Winter"
    if date(2013,3,20) <= d <= date(2013,6,20):
        return "Spring"
    if date(2013,6,21) <= d <= date(2013,9,21):
        return "Summer"
    if date(2013,9,22) <= d <= date(2013,12,20):
        return "Autumn"
    return "Winter"

# ------------------------------------------------------------
# LABELS BY DAY
# ------------------------------------------------------------
labels = []

for d in sorted(hourly["date"].unique()):
    d_date = pd.Timestamp(d).date()
    season = season_2013(d_date)
    wwe = "weekend" if pd.Timestamp(d).weekday() >= 5 else "weekday"

    labels.append({
        "date": d,
        "season": season,
        "weekday_or_weekend": wwe,
        "category": f"{season.lower()}_{wwe}",
    })

labels_df = pd.DataFrame(labels).set_index("date")

f = clusters_dir / "labels_by_day.csv"
labels_df.to_csv(f); written_files.append(f.name)

hourly = hourly.merge(labels_df, left_on="date", right_index=True, how="left")

# ------------------------------------------------------------
# DETECT VRE (EU)
# ------------------------------------------------------------
vre_cols = [c for c in hourly.columns if c.startswith("gen_") and c.endswith("_MW")]
hourly["vre_total_MW"] = hourly[vre_cols].sum(axis=1)

# ------------------------------------------------------------
# COUNTRY × CATEGORY SUMMARY (FINAL)
# ------------------------------------------------------------
MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]
rows = []

for cat, days in labels_df.groupby("category").groups.items():
    hsub = hourly[hourly["date"].isin(days)]

    total_demand_eu = hsub["total_demand_MW"].mean()

    for c in MAIN_COUNTRIES + ["EU"]:

        if c == "EU":
            demand = total_demand_eu
            vre = hsub["vre_total_MW"].mean()

            # demand-weighted EU price
            num = 0.0
            den = 0.0
            for cc in MAIN_COUNTRIES:
                dcol = f"demand_{cc}_MW"
                pcol = f"price_{cc}_EUR_per_MWh"
                if dcol in hsub.columns and pcol in hsub.columns:
                    w = hsub[dcol].mean()
                    num += w * hsub[pcol].mean()
                    den += w
            price = num / den if den > 0 else np.nan

        else:
            dcol = f"demand_{c}_MW"
            pcol = f"price_{c}_EUR_per_MWh"

            demand = hsub[dcol].mean() if dcol in hsub.columns else 0.0
            share = demand / total_demand_eu if total_demand_eu > 0 else 0.0
            vre = hsub["vre_total_MW"].mean() * share

            price = hsub[pcol].mean() if pcol in hsub.columns else np.nan

        rows.append({
            "country": c,
            "category": cat,
            "season": labels_df.loc[days[0], "season"],
            "weekday_or_weekend": labels_df.loc[days[0], "weekday_or_weekend"],
            "n_days": len(days),
            "mean_daily_demand_MW": demand,
            "mean_daily_vre_MW": vre,
            "mean_price_EUR_per_MWh": price,
        })

country_categories = pd.DataFrame(rows)

f = baseline_dir / "baseline_nohub_2013_country_categories.csv"
country_categories.to_csv(f, index=False); written_files.append(f.name)

# ------------------------------------------------------------
# CATEGORY SUMMARY (EU PRICE)
# ------------------------------------------------------------
summary_rows = []

for cat, days in labels_df.groupby("category").groups.items():
    hsub = hourly[hourly["date"].isin(days)]

    summary_rows.append({
        "category": cat,
        "season": labels_df.loc[days[0], "season"],
        "weekday_or_weekend": labels_df.loc[days[0], "weekday_or_weekend"],
        "n_days": len(days),
        "mean_price_EUR_per_MWh": hsub["total_demand_MW"].mean(),
    })

f = clusters_dir / "category_summary.csv"
pd.DataFrame(summary_rows).set_index("category").to_csv(f)
written_files.append(f.name)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ BASELINE M5 COMPLETE — FILES WRITTEN / UPDATED:")
for fn in written_files:
    print(" -", fn)


Loaded 8760 hourly rows

✔ BASELINE M5 COMPLETE — FILES WRITTEN / UPDATED:
 - labels_by_day.csv
 - baseline_nohub_2013_country_categories.csv
 - category_summary.csv


In [20]:
# ============================================================
# FINAL M6 — REBUILD VRE × COUNTRY × CATEGORY (2013)
# BASELINE (NO-HUB) SCENARIO
#
# SOURCE OF TRUTH:
#   - Hourly dispatch → gen_*_MW (from baseline M3)
#   - Category labels → labels_by_day.csv (from M5)
#
# OUTPUT:
#   baseline_nohub_2013_vre_by_category.csv
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
out_root = root / "results" / "daily_runs"

clusters_dir = out_root / "clusters" / "calendar_8cats_2013"
baseline_dir = out_root / "baseline_nohub_2013"
baseline_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD CATEGORY LABELS
# ------------------------------------------------------------

labels = (
    pd.read_csv(
        clusters_dir / "labels_by_day.csv",
        parse_dates=["date"]
    )
    .set_index("date")
)

# ------------------------------------------------------------
# LOAD ALL HOURLY FILES
# ------------------------------------------------------------

hourly_frames = []

for m in range(1, 13):
    mm = f"2013-{m:02d}"
    fn = out_root / mm / f"hourly_power_{mm}.csv"

    if not fn.exists():
        print(f"[WARN] Missing hourly file: {fn}")
        continue

    df = pd.read_csv(fn, parse_dates=["snapshot"])
    df["date"] = df["snapshot"].dt.normalize()
    hourly_frames.append(df)

if not hourly_frames:
    raise RuntimeError("No hourly files found for baseline scenario.")

hourly = pd.concat(hourly_frames, ignore_index=True)

print(f"Loaded {len(hourly)} hourly rows.")

# ------------------------------------------------------------
# DETECT VRE GENERATION COLUMNS (gen_*_MW)
# ------------------------------------------------------------

vre_carriers = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "offwind-float",
    "solar",
    "solar-hsat",
    "ror",
]

vre_cols = [
    c for c in hourly.columns
    if c.startswith("gen_")
    and c.endswith("_MW")
    and any(c == f"gen_{vc}_MW" for vc in vre_carriers)
]

if not vre_cols:
    raise RuntimeError(
        "No VRE generation columns found in hourly data.\n"
        f"Available columns: {list(hourly.columns)}"
    )

print("Detected VRE technologies:", vre_cols)

# ------------------------------------------------------------
# MERGE CATEGORY LABELS
# ------------------------------------------------------------

hourly = hourly.merge(
    labels[["category", "season", "weekday_or_weekend"]],
    left_on="date",
    right_index=True,
    how="inner"
)

# ------------------------------------------------------------
# BUILD VRE × TECHNOLOGY × CATEGORY
# ------------------------------------------------------------

rows = []

for col in vre_cols:
    tech = col.replace("gen_", "").replace("_MW", "")

    for cat, g in hourly.groupby("category"):
        rows.append({
            "technology": tech,
            "category": cat,
            "season": g["season"].iloc[0],
            "weekday_or_weekend": g["weekday_or_weekend"].iloc[0],
            "n_hours": len(g),
            "mean_vre_MW": g[col].mean(),
            "mean_daily_vre_MWh": g[col].mean() * 24.0,
        })

vre_by_category = (
    pd.DataFrame(rows)
    .set_index(["technology", "category"])
    .sort_index()
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

out_fn = baseline_dir / "baseline_nohub_2013_vre_by_category.csv"
vre_by_category.to_csv(out_fn)

print("\n✔ BASELINE M6 COMPLETE — FILE WRITTEN:")
print(" -", out_fn)
print("\nPreview:")
print(vre_by_category.head(12))


Loaded 8760 hourly rows.
Detected VRE technologies: ['gen_offwind-ac_MW', 'gen_offwind-dc_MW', 'gen_offwind-float_MW', 'gen_onwind_MW', 'gen_ror_MW', 'gen_solar_MW', 'gen_solar-hsat_MW']

✔ BASELINE M6 COMPLETE — FILE WRITTEN:
 - C:\Users\franc\pypsa\pypsa-eur\results\daily_runs\baseline_nohub_2013\baseline_nohub_2013_vre_by_category.csv

Preview:
                           season weekday_or_weekend  n_hours   mean_vre_MW  \
technology category                                                           
offwind-ac autumn_weekday  Autumn            weekday     1560  28466.518677   
           autumn_weekend  Autumn            weekend      600  28761.281884   
           spring_weekday  Spring            weekday     1608  25214.042276   
           spring_weekend  Spring            weekend      624  23463.129338   
           summer_weekday  Summer            weekday     1584  18188.209651   
           summer_weekend  Summer            weekend      648  22495.344583   
           winter_

In [ ]:
import pandas as pd
from pathlib import Path

fn = Path(r"C:\Users\franc\pypsa\pypsa-eur\results\daily_runs\2013-01\hourly_power_2013-01.csv")
df = pd.read_csv(fn)
print(df.columns.tolist())


In [18]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\franc\pypsa\pypsa-eur\results\daily_runs\2013-01\hourly_power_2013-01.csv"
)

print(df.columns.tolist())


['snapshot', 'date', 'hour', 'total_demand_MW', 'total_generation_MW', 'demand_DK_MW', 'demand_DE_MW', 'demand_NL_MW', 'demand_NO_MW', 'demand_SE_MW', 'demand_BE_MW', 'demand_FR_MW', 'demand_PL_MW', 'demand_PT_MW', 'generation_DK_MW', 'generation_DE_MW', 'generation_NL_MW', 'generation_NO_MW', 'generation_SE_MW', 'generation_BE_MW', 'generation_FR_MW', 'generation_PL_MW', 'generation_PT_MW', 'price_DK_EUR_per_MWh', 'price_DE_EUR_per_MWh', 'price_NL_EUR_per_MWh', 'price_NO_EUR_per_MWh', 'price_SE_EUR_per_MWh', 'price_BE_EUR_per_MWh', 'price_FR_EUR_per_MWh', 'price_PL_EUR_per_MWh', 'price_PT_EUR_per_MWh', 'load_shed_MW', 'gen_CCGT_MW', 'gen_OCGT_MW', 'gen_biomass_MW', 'gen_coal_MW', 'gen_geothermal_MW', 'gen_lignite_MW', 'gen_load_shedding_MW', 'gen_nuclear_MW', 'gen_offwind-ac_MW', 'gen_offwind-dc_MW', 'gen_offwind-float_MW', 'gen_oil_MW', 'gen_onwind_MW', 'gen_ror_MW', 'gen_solar_MW', 'gen_solar-hsat_MW']
